Viewing the data

In [2]:
import pandas as pd

# Load the dataset
file_path = 'dataset_wellnessdimensions.csv'
data = pd.read_csv(file_path)

# Display the first few rows of the dataset to understand its structure
data.head()

,Unnamed: 0,Text,Aspect,Explanations
0,0,Once it happens there is no going back and you...,4,empty void of darkness
1,1,Now i am filled with guilt and stuck feeling l...,4,filled with guilt and stuck feeling like crap
2,2,And i got home and the voices returned and kep...,4,piece of shit
3,3,"Before her I survived since shit, sexual assau...",1,sexual assaults
4,4,I feel like there are only three things that y...,1,"ugly, dumb, and useless"


Cleaning The Data

In [3]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [4]:
# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Define stop words and punctuation set
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [6]:
# Function to clean and preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove stopwords and punctuation, and lemmatize the words
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and word not in punctuation]
    # Join tokens back into a string
    return ' '.join(cleaned_tokens)

# Apply the preprocessing to the 'Text' and 'Explanations' columns
data['Cleaned_Text'] = data['Text'].apply(preprocess_text)
data['Cleaned_Explanations'] = data['Explanations'].apply(preprocess_text)

# Drop unnecessary columns
cleaned_data = data[['Cleaned_Text', 'Cleaned_Explanations', 'Aspect']]

# Show the cleaned dataset
cleaned_data.head()

,Cleaned_Text,Cleaned_Explanations,Aspect
0,happens going back stuck empty void darkness f...,empty void darkness,4
1,filled guilt stuck feeling like crap,filled guilt stuck feeling like crap,4
2,got home voice returned kept telling piece shit,piece shit,4
3,survived since shit sexual assault like,sexual assault,1
4,feel like three thing absolutely dumb ugly use...,ugly dumb useless,1


Feature Engineering : Converting text into numerical vectors

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=1000)  # Using the top 1000 features

# Fit and transform the cleaned text data
X_tfidf = vectorizer.fit_transform(cleaned_data['Cleaned_Text']).toarray()

# Display the shape of the resulting feature matrix
X_tfidf.shape

(3092, 1000)

Building autoencoder neural network

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [9]:
# Define the dimensions of the input
input_dim = X_tfidf.shape[1]

In [10]:
autoencoder = models.Sequential([
    # Encoding part
    layers.Input(shape=(input_dim,)),
    layers.Dense(512, activation='relu'),  # First encoding layer
    layers.Dense(256, activation='relu'),  # Second encoding layer (latent space)

    # Decoding part
    layers.Dense(512, activation='relu'),  # First decoding layer
    layers.Dense(input_dim, activation='sigmoid')  # Output layer to reconstruct the input
])

In [11]:
# Compile the model
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

In [12]:
# Display the model structure
autoencoder.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 512)               512512    
                                                                 
 dense_1 (Dense)             (None, 256)               131328    
                                                                 
 dense_2 (Dense)             (None, 512)               131584    
                                                                 
 dense_3 (Dense)             (None, 1000)              513000    
                                                                 
Total params: 1288424 (4.91 MB)
Trainable params: 1288424 (4.91 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Training the autoencoder

In [13]:
# Train the autoencoder
history = autoencoder.fit(X_tfidf, X_tfidf, epochs=50, batch_size=64, validation_split=0.2)


Epoch 1/50
39/39 [==============================] - 0s 6ms/step - loss: 0.0762 - val_loss: 9.9354e-04
Epoch 2/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 3/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 4/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 5/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 6/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 7/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 8/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 9/50
39/39 [==============================] - 0s 5ms/step - loss: 9.9798e-04 - val_loss: 9.9354e-04
Epoch 10/50
39/39 [==============================]

Hyperparameter Optimization with Spider Monkey Algorithm (SMO)